# 记忆治理策略
问题：
1. LLM上下文窗口有限
2. 模型会被陈旧或离题的内容分散注意力
3. 带来高昂的token话费

上下文进行管理：对历史记录进行压缩、清理、重组等

## 消息裁剪
### 调用模型前裁剪上下文
控制token用量，通常保留系统初始消息和最近若干消息，或按token数保留末尾内容。


In [4]:
# 基于装饰器实现
# 1、模型的初始化
import os   
from dotenv import load_dotenv
from langchain_qwq import ChatQwen
from rich import print as rprint

custom_profile = {
"max_input_tokens": 128_000 # 最大上下文长度
}

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    profile=custom_profile, # 手动添加的配置项
)

In [ ]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    messages = state["messages"]
    if len(messages) <= 3:
        return None

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES), # 删除指定消息（这里是删除原有全部消息）
            *new_messages,
        ]
    }


agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)

final_response = agent.invoke(
    {"messages": [HumanMessage("告诉我，你是谁？我是谁？")]},
    config,
)

for msg in final_response["messages"]:
    msg.pretty_print()

## 消息删除
消息裁剪强调“在 `模型调用前裁剪` 消息列表，控制模型可以看到的上下文范围”，而消息删除强调 `模型调用完成后` 将某些消息从消息列表中移除 ，永久更改状态。

In [3]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]

    # 保持最近的 5 条消息
    if len(messages) > 5:
        # 框架中通常使用 RemoveMessage 来标记删除，并返回更新状态。
        to_delete = len(messages) - 5
        return {
            "messages": [
                RemoveMessage(id=m.id) for m in messages[:to_delete]
            ]
        }

    return None


agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "你好，我是老王"}, config)
agent.invoke({"messages": "从现在起，你叫小王"}, config)
agent.invoke({"messages": "今天天气不错"}, config)

final_response = agent.invoke(
    {"messages": "告诉我，你是谁？我是谁？"},
    config,
)

for msg in final_response["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

好的，老王！从现在开始我就是小王了。有什么需要帮忙的，或者想聊点啥，您随时吩咐～ 😄
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，老王！今天阳光正好，微风不燥，确实是个难得的好天气。您打算趁这舒服的天气出去溜达溜达、晒晒太阳，还是就在家泡杯茶歇歇脚？有啥安排随时跟我唠唠～ 😄
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

哈哈，这个问题问得还挺有意思的～ 

根据咱们刚才的对话：您一开始叫我“老王”，后来特意叮嘱“从现在起，你叫小王”，所以**我现在就是小王**啦。🐶

至于**您是谁**……您还没来得及告诉我您的名字呢！在咱们的聊天里，您是那位主动打招呼、又给我“升职改名”的朋友。如果您愿意分享一下怎么称呼您，我马上记好；如果不想透露也没关系，我就先按您开头的称呼，尊称您一声**“老王”**或者**“老朋友”**～ 

所以目前的配置是：**我是小王，您是……？** 等您来定！😄


### RemoveMessage底层做了什么？
你在中间件里返回 [RemoveMessage(id=m.id)] 时，你实际上是向框架发送了一个 `删除指令` 。
[历史消息池 (内存中持续存在)]
├── Message(id="1", content="你好，我是老王")
├── Message(id="2", content="...")
└── RemoveMessage(id="1") <-- 这是一个新追加进去的“墓碑”标记

1. 追加“墓碑”标记：框架收到 RemoveMessage(id="1") 后，并不会去内存的数组里把 id="1" 的对象删掉，而是把这个 RemoveMessage 作为一条新记录追加到当前线程的状态历史中。这个RemoveMessage 就像是一个“墓碑”。
   
2. 运行时过滤合并（Reducer）：当下一次你再次调用 agent.invoke 或者大模型要去读取上下文时，框架的内置合并器（Reducer）会把“原始消息”和“墓碑标记”放在一起进行计算：
   原始消息（id:1）+ 墓碑标记（id:1）= 0 (对外隐藏)
在丢给大模型之前，会自动把被标记删除的消息过滤掉。

## 摘要

把早期历史压缩成摘要，再替换原始消息。

消息裁剪和删除都会导致上下文缺失，影响回答质量和用户体验。和它们相比，摘要是更适合长会话的折中方案：保语义，不保原文。官方推荐内置 SummarizationMiddleware 。上一章已有讲解

### 常见问题
1. 摘要会丢失信息么？
    会有一些细节丢失，但是：
   - 重要信息会保留
   - 最近的消息完整保留
   - 对于大部分场景足够
2. 设置最大token数触发摘要的标准是啥？
   - 模型上下文窗口 4k → 设置 3000
   - 模型上下文窗口 8k → 设置 6000
   - 模型上下文窗口 16k → 设置 12000
    为了留一些余量给工具调用和系统提示

3. 摘要成本高么？
    - 摘要只在超过阈值时触发
    - 可以使用便宜的模型
    - 相比传输全部历史，通常更便宜
4. 摘要触发的频率要关注吗？
    要关注，根据监控摘要触发频率，调整阈值
    - 如果频繁触发 提高阈值
    - 如果从不触发 降低阈值

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentState,
    before_model,
    wrap_tool_call,
    after_agent,
)
from langchain.tools.tool_node import ToolCallRequest
from langchain.messages import HumanMessage, AIMessage, ToolMessage
from langchain.tools import tool
from langgraph.runtime import Runtime
from langgraph.types import Command
from typing import Any, Callable
from pydantic import BaseModel, Field


class WeatherInfo(BaseModel):
    """城市天气情况"""

    city: str = Field(description="城市名称")
    temperature: str = Field(description="气温")
    desc: str = Field(description="当日天气概述")


@tool(parse_docstring=True)
def get_weather(city: str):
    """
    获取当日天气

    Args:
        city: 城市名称
    """
    print(f'调用了get_weather{city}')
    return f"[{city}] 今天气温9~16度，万里无云，天气不错适合外出"


@before_model(can_jump_to=["tools"])
def direct_tool_call(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    print('调用了direct_tool_call')
    last_msg = state["messages"][-1]
    if (
        isinstance(last_msg, HumanMessage)
        and "天气" in last_msg.text
        and "北京" in last_msg.text
    ):
        fake_tool_call = AIMessage(
            content="人工构造的消息",
            tool_calls=[
                {
                    "name": "get_weather",
                    "args": {"city": ""},
                    "id": "direct_call_id",
                }
            ],
        )
        return {
            "messages": [fake_tool_call],
            "jump_to": "tools",
        }
    return None


@wrap_tool_call
def first_check(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    print('调用了first_check')
    print("=" * 30, "-> In first_check Middleware <-", "=" * 30)
    print(f"{request.state.get('jump_to', None) = }")
    print(f"{request.state.get('structured_response', None) = }")
    return handler(request)


@after_agent
def final_check(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    print('调用了final_check')
    print("=" * 30, "-> In final_check Middleware <-", "=" * 30)
    for msg in state["messages"]:
        msg.pretty_print()
    print("=" * 30, "-> 消息打印完毕 <-", "=" * 30)
    print(f"{state.get('jump_to', None) = }")
    print(f"{state.get('structured_response', None) = }")
    return None


agent = create_agent(
    model=model,
    response_format=WeatherInfo,
    middleware=[direct_tool_call, first_check, final_check],
    tools=[get_weather],
)

response = agent.invoke({
    "messages": [HumanMessage("请帮我查询北京当日天气")],
})


调用了direct_tool_call
调用了first_check
============================== -> In first_check Middleware <- ==============================
request.state.get('jump_to', None) = 'tools'
request.state.get('structured_response', None) = None
调用了get_weather
调用了direct_tool_call
调用了final_check
============================== -> In final_check Middleware <- ==============================
================================ Human Message =================================

请帮我查询北京当日天气
================================== Ai Message ==================================

人工构造的消息
Tool Calls:
  get_weather (direct_call_id)
 Call ID: direct_call_id
  Args:
    city:
================================= Tool Message =================================
Name: get_weather

[] 今天气温9~16度，万里无云，天气不错适合外出
================================== Ai Message ==================================

北京今天的天气情况如下：

*   **气温**：9 ~ 16℃
*   **概述**：万里无云，阳光充足。
*   **建议**：非常适合外出活动！

请注意早晚温差较大，出门可以适当添衣。
============================== -> 消息打印完毕 <- ========